# ResNet18 full fine-tuning in Google Colab

This notebook starts with ImageNet-pretrained ResNet18 weights and fine-tunes all model parameters on CelebA. Run the cells from top to bottom.

## 1. Mount Google Drive

Before running the notebook, place `celeba.tar.gz` in `MyDrive/face-recognition-data/`.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ARCHIVE = DRIVE_ROOT / 'face-recognition-data' / 'celeba.tar.gz'
RESULTS_DIR = DRIVE_ROOT / 'face-recognition-results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_ARCHIVE.is_file():
    raise FileNotFoundError(
        f'Archive was not found: {DATA_ARCHIVE}. Check the selected Google account.'
    )

print('Dataset archive:', DATA_ARCHIVE)
print('Results directory:', RESULTS_DIR)

## 2. Download the project and dependencies

In [ ]:
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/Bohdan-Kozlo/face-recognition.git'
PROJECT_DIR = Path('/content/face-recognition')

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'mlflow>=3.15,<4',
        'pytorch-metric-learning>=2.9,<3',
    ],
    check=True,
)

os.environ['FACE_RECOGNITION_CHECKPOINTS_DIR'] = str(RESULTS_DIR / 'checkpoints')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

print('Project directory:', PROJECT_DIR)

## 3. Extract CelebA

In [ ]:
DATASET_DIR = PROJECT_DIR / 'data' / 'celeba'
required_files = [
    DATASET_DIR / 'raw' / 'img_align_celeba',
    DATASET_DIR / 'raw' / 'identity_CelebA.txt',
    DATASET_DIR / 'manifests' / 'train.csv',
    DATASET_DIR / 'manifests' / 'validation.csv',
    DATASET_DIR / 'manifests' / 'test.csv',
]

if not all(path.exists() for path in required_files):
    (PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['tar', '-xzf', str(DATA_ARCHIVE), '-C', str(PROJECT_DIR / 'data')],
        check=True,
    )

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'CelebA archive has an unexpected structure: {missing_files}')

print('CelebA is ready:', DATASET_DIR)

## 4. Full fine-tuning of ResNet18

In Colab, select `Runtime → Change runtime type → T4 GPU`. `initialization='imagenet'` downloads pretrained ImageNet weights. `fine_tuning='all'` updates every ResNet18 parameter and the ArcFace loss weights.

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import torch

from config import CELEBA_MANIFESTS_DIR, CELEBA_RAW_DIR
from evaluation import EvaluationConfig, evaluate_checkpoint
from training import TrainingConfig, train

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled in Colab')

print('GPU:', torch.cuda.get_device_name(0))

training_config = TrainingConfig(
    run_name='resnet18-imagenet-all-colab',
    epochs=14,
    batch_size=64,
    num_workers=0,
    device='cuda',
    fine_tuning='all',
    initialization='imagenet',
    validation_batch_limit=None,
    deterministic=True,
)

training_result = train(training_config)
print(training_result)

## 5. Training results

In [ ]:
client = mlflow.MlflowClient()

loss_points = client.get_metric_history(training_result.run_id, 'train/loss')
gap_points = client.get_metric_history(
    training_result.run_id, 'validation/similarity_gap'
)

training_history = pd.DataFrame({
    'epoch': [point.step for point in loss_points],
    'train_loss': [point.value for point in loss_points],
    'validation_similarity_gap': [point.value for point in gap_points],
})
display(training_history.round(4))

training_figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(training_history['epoch'], training_history['train_loss'], marker='o')
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Loss')
axes[0].grid(alpha=0.3)
axes[1].plot(
    training_history['epoch'],
    training_history['validation_similarity_gap'],
    marker='o',
)
axes[1].set(title='Validation similarity gap', xlabel='Epoch', ylabel='Gap')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Validation and test evaluation

The verification threshold is selected only on validation. The test set uses exactly that threshold.

In [ ]:
common_evaluation = {
    'checkpoint_path': training_result.checkpoint_path,
    'dataset_root': CELEBA_RAW_DIR,
    'batch_size': 64,
    'num_workers': 0,
    'device': 'cuda',
}

validation_result = evaluate_checkpoint(
    EvaluationConfig(
        manifest_path=CELEBA_MANIFESTS_DIR / 'validation.csv',
        **common_evaluation,
    )
)
test_result = evaluate_checkpoint(
    EvaluationConfig(
        manifest_path=CELEBA_MANIFESTS_DIR / 'test.csv',
        **common_evaluation,
    ),
    threshold=validation_result.threshold,
)

evaluation_table = pd.DataFrame({
    'validation': validation_result.metrics,
    'test': test_result.metrics,
})
display(evaluation_table.round(4))

scores_figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, name, result in zip(
    axes,
    ['Validation', 'Test'],
    [validation_result, test_result],
):
    axis.hist(result.genuine_scores, bins=30, alpha=0.7, label='Same person')
    axis.hist(result.impostor_scores, bins=30, alpha=0.7, label='Different people')
    axis.axvline(validation_result.threshold, color='black', linestyle='--', label='Threshold')
    axis.set(title=name, xlabel='Cosine similarity', ylabel='Pairs')
    axis.legend()
plt.tight_layout()
plt.show()

## 7. Save the model and results to Google Drive

In [ ]:
import shutil

RUN_DIR = RESULTS_DIR / training_result.run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(training_result.checkpoint_path, RUN_DIR / 'model.pt')
training_history.to_csv(RUN_DIR / 'training_history.csv', index=False)
evaluation_table.to_csv(RUN_DIR / 'evaluation_metrics.csv')
training_figure.savefig(RUN_DIR / 'training_history.png', bbox_inches='tight')
scores_figure.savefig(RUN_DIR / 'score_distributions.png', bbox_inches='tight')

print('Saved results:', RUN_DIR)
for path in sorted(RUN_DIR.iterdir()):
    print('-', path.name)

## Metric summary

- `train_loss`: should generally decrease.
- `validation_similarity_gap`: same-person similarity minus different-person similarity; higher is better.
- `accuracy`, `f1`, `roc_auc`: higher is better.
- `far`, `frr`, `eer`: lower is better.